# Multi-Paradigm Bug Fixing

This notebook includes:
- SentencePiece Tokenizer Training
- T5-small Pretraining
- 4 Unique Paradigms for AI Bug Fixing:
    - fine-tuning with pretraining
    - fine-tuning without pretraining
    - RAG
    - zero-shot
- Evaluation and Cross-Analysis of all Paradigms


## 1. Environment Setup
- Install dependencies
- Mount Google Drive if needed
- Set paths
- Import project modules
- Set seed and device


In [ ]:
import sys
from pathlib import Path

REPO_URL = "https://github.com/sambennett04/multi-paradigm-bug-fixing.git"
REPO_DIR = Path("/content/multi-paradigm-bug-fixing")

# Clone repo if it doesn't exist, otherwise pull latest
if not REPO_DIR.exists():
    !git clone {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} pull

# Add repo root to Python path (so `from src...` works)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

# Install dependencies
!pip install -r {REPO_DIR}/requirements.txt

# (Optional) Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Debug prints (sanity check)
print("Repo dir:", REPO_DIR)
print("src exists:", (REPO_DIR / "src").exists())
print("data_utils exists:", (REPO_DIR / "src" / "data_utils.py").exists())

In [ ]:
from pathlib import Path
import sys

# Repo root (always correct after %cd)
PROJECT_ROOT = Path.cwd()
print("Project root:", PROJECT_ROOT)

#Add repo to sys path so that imported functions resolve correctly
sys.path.append(str(PROJECT_ROOT))

# Persistent storage (Google Drive)
DRIVE_ROOT = Path("/content/drive/MyDrive/multi-paradigm-bug-fixing")
print("Drive root:", DRIVE_ROOT)

DATA_DIR = DRIVE_ROOT / "data"
DATA_DIR.mkdir(parents=True, exist_ok=True)

# Initialize intermediary and final output folders in mounted google drive
OUTPUTS_DIR = DRIVE_ROOT / "outputs"

TOKENIZER_DIR = OUTPUTS_DIR / "tokenizer"
PRETRAIN_MODEL_DIR = OUTPUTS_DIR / "pretrain_model"
FINETUNE_A_DIR = OUTPUTS_DIR / "finetune_a"
FINETUNE_B_DIR = OUTPUTS_DIR / "finetune_b"
PREDICTIONS_DIR = OUTPUTS_DIR / "predictions"
RESULTS_DIR = OUTPUTS_DIR / "results"

# Create directories
for path in [
    TOKENIZER_DIR,
    PRETRAIN_MODEL_DIR,
    FINETUNE_A_DIR,
    FINETUNE_B_DIR,
    PREDICTIONS_DIR,
    RESULTS_DIR,
]:
    path.mkdir(parents=True, exist_ok=True)

print("Outputs will be saved to:", OUTPUTS_DIR)


## 2. Load and Prepare Pretraining Data
- Load CodeSearchNet Java
- Sample ~50K methods
- Extract method bodies
- Preview samples


In [ ]:
import json
from src.data_utils import load_pretraining_data, save_pretraining_data

#load only the train split of CodeSearchNet, take random sample of 50,000 methods and extract their bodies
#Print a summary record of the dataset loading with 3 example methods
pretraining_methods = load_pretraining_data(n_samples=50000, random_seed=42, audit_sample_count=3)

#cache extracted method bodies in a txt file with one flattened function per line
#this format is in line with what SentencePieceTrainer expects in the next section
pretraining_methods_path = DATA_DIR / "pretraining_methods.json"
save_pretraining_data(pretraining_methods_path = pretraining_methods_path, pretraining_methods= pretraining_methods)

## 3. Train / Load Tokenizer
- Train SentencePiece tokenizer once
- Include T5-required special tokens
- Save and reload tokenizer
- Run sanity checks


## 4. Build T5-small Model
- Define T5Config manually
- Initialize model from scratch
- Resize token embeddings
- Verify parameter count


## 5. Pretraining (For Pipeline A)
- Apply span corruption
- Build pretraining dataset
- Train for 3 epochs
- Save final pretrained model


## 6. Load Fine-tuning Dataset
- Load CodeXGLUE bug-fix dataset
- Prepare train / validation / test splits
- Inspect example pair


## 7. Fine-tune Pipeline A (With Pretraining)
- Load final pretrained checkpoint
- Build fine-tuning dataset
- Train on bug-fixing task
- Generate and save test predictions


## 8. Fine-tune Pipeline B (No Pretraining)
- Initialize fresh T5-small model
- Use same tokenizer and config
- Train on bug-fixing task
- Generate and save test predictions


## 9. RAG Baseline (Few-shot)
- Build retrieval index from bug-fix training examples
- Retrieve top-k similar examples
- Construct few-shot prompts
- Run generation and save predictions


## 10. Zero-shot Baseline
- Build zero-shot prompts
- Run Qwen generation
- Save predictions


## 11. Evaluation
- Compute Exact Match
- Compute CodeBLEU
- Compare all four configurations
- Save result table


## 12. Save Final Outputs
- Save metrics
- Save results table
- Preview a few examples for the report
